# YAZEL RECOVAR Integration Demo

This notebook demonstrates how RECOVAR can be used to filter false positive picks from PhaseNet.

- **RECOVAR**: Classifies waveforms to distinguish real earthquakes from noise using learned representations
- **RECOVAR YAZEL Integration**: Uses sliding windows to score PhaseNet picks and filter false positives

#### PhaseNet Configuration:
- **Overlap**: 0.90 (90% overlap between windows)
- **Stacking**: avg (average predictions across overlapping windows)
- **Model**: PhaseNet `instance` pretrained model


In [ ]:
import obspy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
from datetime import datetime

from yazel_integration_sliding import (
    recovar_pick_cleaner_sliding,
    load_recovar_classifier
)
from demo_plotting import get_phasenet_probabilities, plot_side_by_side_comparison, plot_example, plot_threshold_tradeoff
from demo_utils import load_example_picks, print_confusion_matrix

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

### Setup: Load Model and Data

In [2]:
# Configuration
MODEL_PATH = '/mnt/data_a/ege/recovar_models/exp_instance/representation_learning_autoencoder_ensemble/instance/split0/ep19.h5'
PHASENET_THRESHOLD = 0.32
phasenet_pick_dir = f"filtered_phasenet_picks_dir_thr_{PHASENET_THRESHOLD:.2f}"

# Load catalog for ground truth
catalog_path = '/home/boxx/Public/earthquake_model_evaluations/data/SilivriPaper_2019-09-01__2019-11-30/processed_catalogs/kara74a_phase_picks.csv'
catalog = pd.read_csv(catalog_path)
catalog['p_arrival_time'] = pd.to_datetime(catalog['p_arrival_time'])
catalog = catalog[catalog['station'] == 'SLVT']  # Filter for SLVT station only

# Load PhaseNet picks metadata
phasenet_picks = pd.read_csv(f"{phasenet_pick_dir}/metadata.csv")

print(f"Loaded {len(phasenet_picks)} PhaseNet picks")
print(f"Loaded {len(catalog)} catalog picks for station SLVT")

Loaded 2175 PhaseNet picks
Loaded 534 catalog picks for station SLVT


In [3]:
# Load RECOVAR classifier
print("Loading RECOVAR classifier...")
classifier = load_recovar_classifier(MODEL_PATH)
print("Classifier loaded successfully!")

Loading RECOVAR classifier...


2025-11-28 15:36:07.390323: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22286 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:19:00.0, compute capability: 8.6
2025-11-28 15:36:07.391350: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 15929 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:1a:00.0, compute capability: 8.6
2025-11-28 15:36:07.392196: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 16953 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:67:00.0, compute capability: 8.6
2025-11-28 15:36:07.393843: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 654 MB memory:  -> device: 3, name: NVIDIA GeForce RTX 3090, 

Classifier loaded successfully!


## Select Example Picks

For this demonstration, we'll select:
1. **TRUE PICKS**: PhaseNet picks that match catalog events (real earthquakes)
2. **FALSE PICKS**: PhaseNet picks without catalog events (noise/artifacts)

In [ ]:
# Load and categorize picks into TRUE and FALSE examples
tp_examples, fp_examples = load_example_picks(phasenet_pick_dir, catalog)

print(f"Found {len(tp_examples)} TRUE PICK examples")
print(f"Found {len(fp_examples)} FALSE PICK examples")

**Note:** PhaseNet probability arrays are saved directly in the MSEED files alongside waveform data during the picking phase.

## Visualization

### TRUE PICK Examples (Real Earthquake)

## RECOVAR Filtering Examples

Demonstrating RECOVAR's filtering performance with threshold = 0.07 (max score):
- **TRUE PICK + Kept**: Real earthquake that RECOVAR correctly kept
- **TRUE PICK + Filtered**: Real earthquake that RECOVAR incorrectly rejected  
- **FALSE PICK + Filtered**: Noise/artifact that RECOVAR correctly rejected
- **FALSE PICK + Kept**: Noise/artifact that RECOVAR incorrectly kept

In [11]:
# Process all examples
tp_results = []
fp_results = []

print("Processing TRUE PICKS...")
for example in tp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    tp_results.append(result)

print("Processing FALSE PICKS...")
for example in fp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    fp_results.append(result)

# Extract scores
tp_mean_scores = [r['mean_score'] for r in tp_results]
tp_max_scores = [r['max_score'] for r in tp_results]
fp_mean_scores = [r['mean_score'] for r in fp_results]
fp_max_scores = [r['max_score'] for r in fp_results]

print(f"\nProcessed {len(tp_results)} TRUE PICKS and {len(fp_results)} FALSE PICKS")

Processing TRUE PICKS...
Processing FALSE PICKS...

Processed 3 TRUE PICKS and 46 FALSE PICKS


In [ ]:
# Find examples of each category using the threshold of 0.07
RECOVAR_THRESHOLD = 0.07

# Categorize examples
tp_kept = []  # TRUE PICK kept by RECOVAR
tp_filtered = []  # TRUE PICK filtered by RECOVAR
fp_kept = []  # FALSE PICK kept by RECOVAR
fp_filtered = []  # FALSE PICK filtered by RECOVAR

for example in tp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    example['recovar_result'] = result
    example['phasenet_result'] = get_phasenet_probabilities(example['stream'])
    
    if result['max_score'] >= RECOVAR_THRESHOLD:
        tp_kept.append(example)
    else:
        tp_filtered.append(example)

for example in fp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    example['recovar_result'] = result
    example['phasenet_result'] = get_phasenet_probabilities(example['stream'])
    
    if result['max_score'] >= RECOVAR_THRESHOLD:
        fp_kept.append(example)
    else:
        fp_filtered.append(example)

print(f"TRUE PICKS kept by RECOVAR: {len(tp_kept)}")
print(f"TRUE PICKS filtered by RECOVAR: {len(tp_filtered)}")
print(f"FALSE PICKS kept by RECOVAR: {len(fp_kept)}")
print(f"FALSE PICKS filtered by RECOVAR: {len(fp_filtered)}")

# Show multiple examples - different categories side by side
num_examples_to_show = 2

# Correct decisions: TRUE PICKS KEPT vs FALSE PICKS FILTERED
if tp_kept and fp_filtered:
    print("\n" + "="*80)
    print("CORRECT DECISIONS: TRUE PICKS KEPT (left) vs FALSE PICKS FILTERED (right)")
    print("="*80)
    for i in range(min(num_examples_to_show, len(tp_kept), len(fp_filtered))):
        print(f"\nExample pair {i+1}")
        plot_side_by_side_comparison(
            tp_kept[i], tp_kept[i]['recovar_result'], tp_kept[i]['phasenet_result'],
            fp_filtered[i], fp_filtered[i]['recovar_result'], fp_filtered[i]['phasenet_result'],
            threshold=RECOVAR_THRESHOLD
        )

# Incorrect decisions: TRUE PICKS FILTERED vs FALSE PICKS KEPT
if tp_filtered and fp_kept:
    print("\n" + "="*80)
    print("INCORRECT DECISIONS: TRUE PICKS FILTERED (left) vs FALSE PICKS KEPT (right)")
    print("="*80)
    for i in range(min(num_examples_to_show, len(tp_filtered), len(fp_kept))):
        print(f"\nExample pair {i+1}")
        plot_side_by_side_comparison(
            tp_filtered[i], tp_filtered[i]['recovar_result'], tp_filtered[i]['phasenet_result'],
            fp_kept[i], fp_kept[i]['recovar_result'], fp_kept[i]['phasenet_result'],
            threshold=RECOVAR_THRESHOLD
        )

# If we only have one type of incorrect decision, show it anyway
if tp_filtered and not fp_kept:
    print("\n" + "="*80)
    print("FALSE NEGATIVES: TRUE PICKS FILTERED BY RECOVAR")
    print("="*80)
    for i in range(min(num_examples_to_show, len(tp_filtered))):
        print(f"\nExample {i+1}")
        plot_example(tp_filtered[i], tp_filtered[i]['recovar_result'], 
                    tp_filtered[i]['phasenet_result'], threshold=RECOVAR_THRESHOLD)

if fp_kept and not tp_filtered:
    print("\n" + "="*80)
    print("FALSE POSITIVES: FALSE PICKS KEPT BY RECOVAR")
    print("="*80)
    for i in range(min(num_examples_to_show, len(fp_kept))):
        print(f"\nExample {i+1}")
        plot_example(fp_kept[i], fp_kept[i]['recovar_result'], 
                    fp_kept[i]['phasenet_result'], threshold=RECOVAR_THRESHOLD)

### Performance Metrics at Recommended Threshold (0.07)

In [ ]:
# Display confusion matrix at recommended threshold
print_confusion_matrix(tp_kept, tp_filtered, fp_kept, fp_filtered, RECOVAR_THRESHOLD)

### Threshold Trade-off Analysis

This plot shows how different thresholds affect the filtering performance:
- **Missed True Positives**: Real earthquakes that would be filtered out (False Negatives)
- **Filtered False Positives**: Noise that would be correctly removed (True Negatives)

In [ ]:
# Plot threshold trade-off analysis
plot_threshold_tradeoff(tp_max_scores, fp_max_scores, RECOVAR_THRESHOLD)

## Summary
- A well-chosen RECOVAR threshold can filter many false positives while retaining most true positives

For batch processing and full evaluation, see `run_yazel_batch_sliding.py`